# 03 - LSTM and GRU

The same information sets and metrics as notebook 02, with one-layer sequence
models over a 20-day lookback (paper settings: 32 units, dropout 0.2, Adam with
learning rate 0.001, batch size 64, 8 epochs; see `config.py`). Results are written
to `results/tables/` and appear as the bottom two rows of Figures 4 and 5.

Needs TensorFlow (`pip install -r requirements-dl.txt`). Neural-network training is
not bit-for-bit reproducible across hardware and TensorFlow versions, so re-running
gives numbers close to, but not identical with, the stored results.

Leakage controls: imputation and scaling are fitted on training rows only, the
split is chronological, and the validation set is the last 15% of the training period.

In [ ]:
# Make the project package importable, from this folder or the project root.
# In Colab: mount Drive and set ROOT = Path("/content/drive/MyDrive/NEPSE_replication") instead.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "nepse").is_dir())
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

In [ ]:
from nepse import config, features, experiments

config.ensure_dirs()
ds = features.build_dataset()
print(f"lookback={config.LOOKBACK} | train {int(ds.train_mask.sum())} "
      f"| test {int((~ds.train_mask).sum())}")

In [ ]:
dl_metrics, dl_predictions = experiments.run_deep_learning(ds, cells=("LSTM", "GRU"))
experiments.save_deep(dl_metrics, dl_predictions)

In [ ]:
print(experiments.metric_matrix(dl_metrics, "RMSE_bps")
      .rename(columns=config.INFO_LABELS).round(2).to_string())